# Hotel Booking Cancellation — Exploratory Data Analysis

This notebook contains the initial exploratory data analysis (EDA) of the hotel booking dataset. The goal is to understand the dataset, target variable, missing values, duplicates, important patterns, and suspicious data values before building the machine learning pipeline.

In [ ]:
import pandas as pd

df = pd.read_csv("data/hotel_bookings.csv")

## 1. Initial Dataset Inspection

In [ ]:
# Display the first 5 rows of the dataset.
# This gives us an initial look at what the data looks like
# and what kind of information each booking contains.
print(df.head())

# Display the shape of the dataset.
# Shape returns: (number of rows, number of columns).
# Here we have 119,390 bookings and 32 columns.
print("\nShape:")
print(df.shape)

# Display the names of all columns in the dataset.
# This helps us understand what information is available
# for each hotel booking.
print("\nColumns:")
print(df.columns.tolist())

# Display general information about the dataset.
# This shows:
# - number of rows
# - column names
# - number of non-missing values
# - data types (int, float, object, etc.)
# We use this to understand the structure of our data before doing any preprocessing or machine learning.
print("\nDataset Info:")
df.info()

## 2. Missing Values

In [ ]:
# Count the number of missing values in each column.
# Missing values are important because ML models generally cannot directly work with missing data.
# We found missing values mainly in:
# children, country, agent and company.
# We will investigate these before deciding how to handle them.
print("\nMissing Values:")
print(df.isnull().sum())

## 3. Duplicate Analysis

In [ ]:
# Count exact duplicate rows in the dataset.
# Pandas checks whether an entire row is identical to another row across all columns.
# We found 31,994 duplicate rows.
# We should NOT automatically delete them yet because identical rows do not necessarily mean the same booking.
# We need to investigate what these duplicates represent first.
print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nDuplicate rows by cancellation status:")
print(df[df.duplicated(keep=False)]["is_canceled"].value_counts())

print("\nDuplicate percentage by class:")

duplicates = df.duplicated(keep=False)

print(
    df[duplicates]["is_canceled"]
    .value_counts(normalize=True) * 100
)

# "If we removed exact duplicates, how much would our dataset shrink?"
# We are NOT removing them yet because identical rows do not necessarily represent erroneous records.
# Different bookings could potentially have identical feature values.
print("\nDataset shape after removing exact duplicates:")
print(df.drop_duplicates().shape)

## 4. Target Variable — Cancellation Distribution

In [ ]:
# Check how many bookings were cancelled vs not cancelled.
# is_canceled is our TARGET variable:
# 0 = booking was not cancelled
# 1 = booking was cancelled
# This tells us how our target classes are distributed.
print("\nCancellation Distribution:")
print(df["is_canceled"].value_counts())

# Convert the cancellation counts into percentages.
# This helps us understand whether our target classes are balanced.
# Approximately:
# 62.96% → not cancelled
# 37.04% → cancelled
# The classes are somewhat imbalanced, but NOT extremely imbalanced
# like the credit card fraud dataset we originally considered.
print("\nCancellation Percentage:")
print(df["is_canceled"].value_counts(normalize=True) * 100)

## 5. Cancellation Rate by Hotel Type

In [ ]:
print("\nCancellation rate by hotel type:")
print(
    df.groupby("hotel")["is_canceled"].mean() * 100
)

## 6. Cancellation Rate by Lead Time

In [ ]:
print("\nCancellation rate by lead time:")

# Group bookings into meaningful lead-time ranges.
# This helps us see whether booking far in advance is associated with a higher or lower cancellation rate.
df["lead_time_group"] = pd.cut(
    df["lead_time"],
    bins=[-1, 7, 30, 90, 180, 365, float("inf")],
    labels=[
        "0-7 days",
        "8-30 days",
        "31-90 days",
        "91-180 days",
        "181-365 days",
        "365+ days"
    ]
)

print(
    df.groupby("lead_time_group", observed=True)["is_canceled"].mean() * 100
)

## 7. Cancellation Rate by Deposit Type

In [ ]:
print("\nCancellation rate by deposit type:")

print(
    df.groupby("deposit_type")["is_canceled"].mean() * 100
)

print("\nBooking count by deposit type:")

print(
    df["deposit_type"].value_counts()
)

print("\nDeposit type vs cancellation:")

print(
    pd.crosstab(
        df["deposit_type"],
        df["is_canceled"],
        normalize="index"
    ) * 100
)

## 8. Cancellation Rate by Previous Cancellations

In [ ]:
print("\nCancellation rate by previous cancellations:")

print(
    df.groupby("previous_cancellations")["is_canceled"].mean() * 100
)

## 9. Cancellation Rate by Special Requests

In [ ]:
print("\nCancellation rate by number of special requests:")

print(
    df.groupby("total_of_special_requests")["is_canceled"].mean() * 100
)

## 10. Cancellation Rate by Market Segment

In [ ]:
print("\nCancellation rate by market segment:")

print(
    df.groupby("market_segment")["is_canceled"].mean() * 100
)

## 11. Numerical Feature Statistics

In [ ]:
print("\nBasic statistics for numerical features:")
print(df.describe().T)

## 12. Data Quality / Suspicious Value Checks

In [ ]:
print("\nROWS CHECK")

print("\nSuspicious adult counts:")
print(df[df["adults"] > 10][
    ["hotel", "adults", "children", "babies", "is_canceled"]
])

print("\nSuspicious ADR values:")
print(df[df["adr"] < 0][
    ["hotel", "adr", "is_canceled"]
])

print("\nHighest ADR values:")
print(
    df.nlargest(10, "adr")[
        ["hotel", "adr", "adults", "children", "is_canceled"]
    ]
)

## EDA Status

The initial EDA is complete enough to move into data cleaning and preprocessing. Important findings and preprocessing decisions will be implemented in the ML pipeline rather than modifying the original dataset inside this notebook.